In [ ]:
suppressPackageStartupMessages({
  library(Seurat)
  library(ggplot2)
  library(dplyr)
  library(tidyr)
  library(patchwork)
  library(ggridges)
  library(ComplexHeatmap)
  library(circlize)
  library(viridis)
  library(ggnewscale)
  library(grid)
  library(scales)
})
root <- getwd()
marco <- readRDS(file.path(root, "Affirmseq_marco.rds"))
fd <- marco@misc$figure_data
theme_set(theme_classic(base_size = 14))

In [ ]:
m <- fd$matrix_meta
p_n <- ggplot(m, aes(x, y, color = nuclei_clip)) + geom_point(size = 0.8) + scale_color_viridis_c(option = "magma", name = "Nuclei intensity") + coord_fixed() + scale_y_reverse() + labs(title = "Cell nuclei matrix", x = "Coordinate X", y = "Coordinate Y") + theme(plot.background = element_rect(fill = "black"), panel.background = element_rect(fill = "black"), axis.text = element_text(color = "white"), axis.title = element_text(color = "white"), plot.title = element_text(color = "white", size = 10), legend.text = element_text(color = "white"), legend.title = element_text(color = "white", size = 9), legend.background = element_rect(fill = "black"))
p_e <- ggplot(m, aes(x, y, color = ecoli_clip)) + geom_point(size = 0.8) + scale_color_viridis_c(option = "magma", name = expression(italic(E.~coli)~intensity)) + coord_fixed() + scale_y_reverse() + labs(title = expression(italic(E.~coli)~matrix), x = "Coordinate X", y = "Coordinate Y") + theme(plot.background = element_rect(fill = "black"), panel.background = element_rect(fill = "black"), axis.text = element_text(color = "white"), axis.title = element_text(color = "white"), plot.title = element_text(color = "white", size = 10), legend.text = element_text(color = "white"), legend.title = element_text(color = "white", size = 9), legend.background = element_rect(fill = "black"))
ggsave(file.path(root, "Affirmseq_marco_matrix_panels.pdf"), p_n | p_e, width = 8.2, height = 3.2, device = cairo_pdf)

p_status <- ggplot(fd$ecoli_status, aes(cell_type, freq, fill = Ecoli_Status)) + geom_col(width = 0.58) + scale_fill_manual(values = c("Negative" = "grey80", "Positive" = "#377EB8"), name = expression(italic(E.~coli)~status)) + scale_y_continuous(labels = percent, expand = c(0, 0)) + scale_x_discrete(labels = c(background = "Cell nuclei\nnegative", cell = "Cell nuclei\npositive")) + labs(x = NULL, y = "Percentage (%)") + theme(axis.text = element_text(color = "black"), legend.position = "right")
ggsave(file.path(root, "Affirmseq_marco_Ecoli_status.pdf"), p_status, width = 3.4, height = 3.2, device = cairo_pdf)

u <- fd$umap
p_bin <- ggplot(u, aes(Phago_Bin, ecoli_log2, fill = Phago_Bin)) + geom_violin(scale = "width", trim = FALSE, alpha = 0.85, linewidth = 0.35) + geom_boxplot(width = 0.14, fill = "white", outlier.shape = NA, alpha = 0.85) + scale_fill_manual(values = fd$bin_colors) + labs(x = "Phagocytosis group", y = expression(italic(E.~coli)~signal~(log[2]))) + guides(fill = "none") + theme(axis.text.x = element_text(angle = 45, hjust = 1, color = "black"), axis.text.y = element_text(color = "black"))
ggsave(file.path(root, "Affirmseq_marco_pBin_Ecoli.pdf"), p_bin, width = 5.2, height = 4.2, device = cairo_pdf)

p_qc <- ggplot(fd$qc_long, aes(Phago_Bin, Value, fill = Phago_Bin)) + geom_boxplot(outlier.size = 0.25, alpha = 0.8, linewidth = 0.35) + facet_wrap(~Metric, ncol = 1, scales = "free_y") + scale_fill_manual(values = fd$bin_colors) + labs(x = "Phagocytosis group", y = NULL) + guides(fill = "none") + theme(axis.text.x = element_text(angle = 45, hjust = 1, color = "black"), axis.text.y = element_text(color = "black"), strip.background = element_blank())
ggsave(file.path(root, "Affirmseq_marco_QC_by_bin.pdf"), p_qc, width = 5.2, height = 6.2, device = cairo_pdf)

p_knn <- ggplot(fd$knn_sensitivity, aes(k, effect_size)) + geom_line(linewidth = 0.6) + geom_point(size = 2) + geom_vline(xintercept = 10, linetype = "dashed") + labs(x = "k (Number of nearest neighbors)", y = "Effect size (Cliff's delta)") + theme(axis.text = element_text(color = "black"))
ggsave(file.path(root, "Affirmseq_marco_KNN_sensitivity.pdf"), p_knn, width = 4.8, height = 3.1, device = cairo_pdf)

In [ ]:
u <- fd$umap
cl <- fd$cluster_levels
cr <- fd$cluster_rank_levels
cc <- fd$cluster_cols
u$seurat_clusters <- factor(as.character(u$seurat_clusters), levels = cl)
fd$cluster_mean$seurat_clusters <- factor(as.character(fd$cluster_mean$seurat_clusters), levels = cr)
p_umap <- ggplot(u, aes(umap_1, umap_2, color = seurat_clusters)) + geom_point(size = 0.8) + scale_color_manual(values = cc, limits = cl, breaks = cl) + coord_fixed() + labs(x = "UMAP 1", y = "UMAP 2", color = NULL) + theme(axis.text = element_text(color = "black"))
p_density <- ggplot(u, aes(ecoli_log2, fill = Phago_Bin)) + geom_density(alpha = 0.75, color = NA) + scale_fill_manual(values = fd$bin_colors, name = "Groups") + labs(x = expression(italic(E.~coli)~signal~intensity~(log[2])), y = "Cell density") + theme(axis.text = element_text(color = "black"))
p_mean <- ggplot(fd$cluster_mean, aes(seurat_clusters, mean, color = seurat_clusters)) + geom_errorbar(aes(ymin = mean - se, ymax = mean + se), width = 0.1) + geom_point(size = 2.4) + geom_smooth(aes(group = 1), method = "lm", se = FALSE, linetype = "dashed", color = "black", linewidth = 0.4) + scale_color_manual(values = cc, limits = cr, breaks = cr) + scale_x_discrete(limits = cr) + labs(x = "Cluster", y = expression(Mean~italic(E.~coli)~signal~intensity~(log[2]))) + guides(color = "none") + theme(axis.text = element_text(color = "black"))
p_ecoli_umap <- ggplot(u, aes(umap_1, umap_2, color = ecoli_log2)) + geom_point(size = 0.7) + scale_color_viridis_c(option = "D", name = expression(italic(E.~coli)~density)) + coord_fixed() + labs(x = "UMAP 1", y = "UMAP 2") + theme(axis.text = element_text(color = "black"))
ggsave(file.path(root, "Affirmseq_marco_UMAP_density_mean.pdf"), (p_density | p_umap | p_mean | p_ecoli_umap), width = 12.5, height = 3.2, device = cairo_pdf)

ph <- fd$phago_heatmap
ph_t <- t(ph)
row_split <- factor(ifelse(rownames(ph_t) == "Negative", "Neg", "Pos"), c("Neg", "Pos"))
col_split <- factor(fd$phago_gene_group[colnames(ph_t)], c("Inhibited (Down)", "Response (Up)"))
ha_left <- rowAnnotation(`E. coli` = fd$phago_bin_ecoli[rownames(ph_t)], col = list(`E. coli` = colorRamp2(seq(min(fd$phago_bin_ecoli), max(fd$phago_bin_ecoli), length = 20), viridis(20))))
ha_top <- HeatmapAnnotation(Module = col_split, col = list(Module = c("Inhibited (Down)" = "#5DADE2", "Response (Up)" = "#E74C3C")))
ht <- Heatmap(ph_t, name = "Z-score", col = colorRamp2(c(-2, 0, 2), c("#5DADE2", "white", "#E74C3C")), row_split = row_split, column_split = col_split, cluster_rows = FALSE, cluster_columns = TRUE, top_annotation = ha_top, left_annotation = ha_left, row_names_gp = gpar(fontsize = 8), column_names_gp = gpar(fontsize = 6), column_title = "Phagocytosis Progression", border = TRUE)
cairo_pdf(file.path(root, "Affirmseq_marco_phago_heatmap.pdf"), width = 12, height = 4.8); draw(ht, merge_legend = TRUE); dev.off()


In [ ]:
roi_i <- fd$spatial_roi_infected
roi_b <- fd$spatial_roi_bystanders
p_sp <- ggplot() + stat_density_2d(data = roi_i, aes(x = x, y = y, fill = after_stat(level)), geom = "polygon", h = c(3.5, 3.5), n = 600, alpha = 0.68, color = NA) + scale_fill_viridis_c(option = "viridis", name = "Bacterial\nDensity") + ggnewscale::new_scale_fill() + geom_point(data = roi_b, aes(x = x, y = y, fill = n_high_neighbors), shape = 23, color = "white", stroke = 0.8, size = 3.6) + scale_fill_gradientn(colors = c("#2166ac", "#f7f7f7", "#b2182b"), limits = c(2.5, 10), oob = squish, name = "Bystander\nStress") + coord_fixed(xlim = fd$spatial_view$x, ylim = rev(fd$spatial_view$y), expand = FALSE) + labs(x = "Spatial X", y = "Spatial Y") + theme_classic(base_size = 13) + theme(plot.background = element_rect(fill = "black"), panel.background = element_rect(fill = "black"), legend.background = element_rect(fill = "black"), legend.key = element_rect(fill = "black"), legend.text = element_text(color = "white"), legend.title = element_text(color = "white", face = "bold"), axis.line = element_line(color = "white"), axis.ticks = element_line(color = "white"), axis.text = element_text(color = "white"), axis.title = element_text(color = "white", face = "bold"))
ggsave(file.path(root, "Affirmseq_marco_spatial_bystander.pdf"), p_sp, width = 6.4, height = 4.8, device = cairo_pdf)

bh <- fd$bystander_heatmap
gm <- factor(fd$bystander_gene_module[rownames(bh)], c("Infected-specific", "Bystander-specific", "Gradient"))
ct <- factor(ifelse(grepl("Infected", colnames(bh)), "Infected", "Bystander"), c("Infected", "Bystander"))
stress_cols <- c(setNames(viridis(3), paste0("Infected-", 1:3)), setNames(colorRampPalette(c("#2166ac", "#f7f7f7", "#b2182b"))(5), paste0("Bystander-", 1:5)))
top_ha <- HeatmapAnnotation(`Phagocytosis status` = ct, `Intracellular bacterial burdens` = colnames(bh), col = list(`Phagocytosis status` = c("Infected" = "#E74C3C", "Bystander" = "#3498DB"), `Intracellular bacterial burdens` = stress_cols), annotation_name_side = "left")
right_ha <- rowAnnotation(`Gene module` = gm, col = list(`Gene module` = c("Infected-specific" = "#D32F2F", "Bystander-specific" = "#1976D2", "Gradient" = "#F57C00")))
label_genes <- c("Nlrx1", "Vps18", "Edem1", "Trib3", "Lamp2", "Psma1", "Wdr5", "Top1", "Gapdh", "Pgk1", "Glrx2", "Vsir", "Hax1")
labs <- ifelse(rownames(bh) %in% label_genes, rownames(bh), "")
ht2 <- Heatmap(bh, name = "Z-score", col = colorRamp2(c(-2, 0, 2), c("#5DADE2", "white", "#E74C3C")), row_split = gm, column_split = ct, cluster_rows = TRUE, cluster_columns = FALSE, show_row_names = TRUE, row_labels = labs, row_names_gp = gpar(fontsize = 10, fontface = "italic"), column_names_gp = gpar(fontsize = 8), column_names_rot = 45, top_annotation = top_ha, right_annotation = right_ha, row_gap = unit(3, "mm"), column_gap = unit(3, "mm"), column_title = "Phagocytosis Stress Response Gradient", border = TRUE)
cairo_pdf(file.path(root, "Affirmseq_marco_bystander_heatmap.pdf"), width = 7.6, height = 10.5); draw(ht2, merge_legend = TRUE, heatmap_legend_side = "right", annotation_legend_side = "right"); dev.off()